# CitiBike Intelligent Mobility — Proyecto integrado U1 + U2 + U3

**Cristhian Chuquitarqui Chura · Análisis espacial · Data Geniuses** · Big Data lambda26

> **Pregunta central:** ¿dónde se concentra la actividad de Citi Bike NYC, hoy y en la siguiente ventana de 15 minutos?
> **Historia:** **ANALIZAR EL PASADO** (U1 · batch) → **ANTICIPAR EL FUTURO** (U2 · streaming) → **COMPROBAR EL FUNCIONAMIENTO** (U3 · integración)

Este único notebook contiene **todo el código de las tres unidades**, en orden. Ejecútalo con **Run All** (≈ 20 min) o sección por sección.
Mientras corre, el **dashboard general de Grafana** (http://localhost:43000/d/citibike-general-cristhian · admin/admin) se va llenando.

| Sección | Qué hace | Tiempo aprox. |
|---|---|---|
| **0. Sesión Spark** | Sesión con el conector de Kafka; entre unidades se cierra y se abre otra para liberar memoria | 1 min |
| **01. U1 · Batch** | Bronze → Silver → Gold (Parquet), zonas de 0,02°, ranking y modelo de alta actividad (MLlib) | 5 min |
| **02. U2 · Streaming** | Parte A: modelo de series de tiempo t+1 por zona. Parte B: Kafka + Spark Structured Streaming con inferencia en vivo | 3 + 6 min |
| **03. U3 · Integración** | Ejecución integrada con compuertas, KPI en Grafana, manifiesto, prueba de fallo y no regresión | 5 min |
| **04. Cierre** | Regenera el dashboard general con los resultados de U1 y deja el streaming en vivo para Grafana | 1 min |

**Antes de empezar:** `docker compose up -d --build --wait` y los 5 CSV en `datos/` (`docker compose exec spark python datos/descargar_datos.py`).

# 0. Sesión Spark y parámetros de U1

Sesión de U1 (ya incluye el conector de Kafka, 4 GB). Entre unidades se cierra la sesión para liberar memoria y cada parte de U2 abre la suya con su propia celda.

In [1]:
import json, os, datetime as dt
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

spark = (SparkSession.builder.appName("U1_Batch_Espacial_CristhianChuquitarqui").master("local[*]")
         .config("spark.driver.memory", "4g").config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.session.timeZone", "America/New_York")      # los CSV están en hora local de NYC
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")   # conector Kafka para U2
         .config("spark.driver.extraJavaOptions", "-Xss16m")   # pila amplia para los hilos de streaming
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

DATOS = "../datos/*.csv"                 # los 5 CSV de julio 2026
SALIDA = "../salidas/u1"
TAM_CELDA, LAT0, LON0 = 0.02, 40.50, -74.30   # cuadrícula de zonas: celdas de 0,02° (~2 km) desde la esquina SO de NYC
os.makedirs(SALIDA, exist_ok=True)
print("Spark", spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9a0c8c97-d62a-47c5-bc72-ec751c85b527;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central


	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central


	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 973ms :: artifacts dl 12ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 from central in [default]
	org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 from central in [default]
	org.slf4j#slf4j-api;2.0.17 from central in [default]
	org.xerial.snappy#snappy-java

26/11/20 05:56:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 4.2.0


---
# 01 · UNIDAD I — ANÁLISIS HISTÓRICO (BATCH)

**¿Qué zonas concentran la actividad y cuáles tienen mayor probabilidad de alta actividad?**

## U1 · Batch — ETL de viajes y zonas de alta actividad

**Integrante:** Cristhian Chuquitarqui Chura · **Equipo:** Data Geniuses · **Proyecto Sello:** CitiBike Intelligent Mobility · **Dimensión D:** patrones espaciales

**Pregunta U1:** *¿Qué zonas concentran la actividad de Citi Bike y cuáles tienen mayor probabilidad de alta actividad?*

| Sesión | Qué se aplica aquí | Sección |
|---|---|---|
| S1 · Arquitectura Lambda | Este notebook es la **capa batch**: lo que produce (Silver, mapa de zonas) lo usa la capa de streaming de U2 | todo |
| S2 · Fundamentos PySpark | Transformaciones, agregaciones, evaluación perezosa (`explain`) | 2, 3 |
| S3 · Calidad y particionado | Perfil de calidad, reglas Silver, Parquet particionado por `fecha` | 2, 3, 5 |
| S4 · ML con Spark MLlib (regresión) | Regresión de la actividad por zona y hora, contra un baseline | 6, 7 |

**Salidas** (carpeta `salidas/u1/`): `silver/`, `estaciones_zona/` + `estaciones_zona.json`, `gold_zona_hora/`, `gold_ranking_zonas/`, `modelo_actividad_zona/`, `metricas_u1.json`.

### 2. Bronze: lectura con esquema explícito y perfil de calidad

Los IDs de estación (`6140.05`) se leen como texto: si Spark los infiere como número pierde ceros y decimales.

In [2]:
esquema = StructType([
    StructField("ride_id", StringType()), StructField("rideable_type", StringType()),
    StructField("started_at", TimestampType()), StructField("ended_at", TimestampType()),
    StructField("start_station_name", StringType()), StructField("start_station_id", StringType()),
    StructField("end_station_name", StringType()), StructField("end_station_id", StringType()),
    StructField("start_lat", DoubleType()), StructField("start_lng", DoubleType()),
    StructField("end_lat", DoubleType()), StructField("end_lng", DoubleType()),
    StructField("member_casual", StringType())])
bronze = (spark.read.option("header", True).option("timestampFormat", "yyyy-MM-dd HH:mm:ss[.SSS]")
          .schema(esquema).csv(DATOS))

duracion = (F.unix_timestamp("ended_at") - F.unix_timestamp("started_at")) / 60.0
en_nyc = lambda la, lo: F.col(la).between(40.45, 40.95) & F.col(lo).between(-74.30, -73.65)
falla = lambda cond: F.sum(F.when(F.coalesce(cond, F.lit(False)), 1).otherwise(0))

perfil = bronze.agg(
    F.count("*").alias("filas"),
    F.countDistinct("ride_id").alias("ride_id_unicos"),
    falla(F.col("start_station_id").isNull() | F.col("end_station_id").isNull()).alias("sin_estacion"),
    falla(F.col("started_at").isNull() | F.col("ended_at").isNull()).alias("sin_fecha"),
    falla(~duracion.between(1.0, 180.0)).alias("duracion_fuera_1_180_min"),
    falla(~(en_nyc("start_lat", "start_lng") & en_nyc("end_lat", "end_lng"))).alias("coordenadas_fuera_nyc"),
).first().asDict()
perfil["duplicados_ride_id"] = perfil["filas"] - perfil["ride_id_unicos"]
for k, v in perfil.items():
    print(f"{k:28s} {v:>12,}")

filas                           4,993,137
ride_id_unicos                  4,993,137
sin_estacion                       17,730
sin_fecha                               0
duracion_fuera_1_180_min            5,660
coordenadas_fuera_nyc                   0
duplicados_ride_id                      0


### 3. Silver: reglas de calidad y Parquet particionado por fecha

Reglas comunes del equipo: sin duplicados de `ride_id`, sin estación nula, duración entre 1 y 180 min y coordenadas dentro de NYC.
Spark es **perezoso**: hasta el `write` solo se arma el plan (`explain`); el trabajo se ejecuta una vez, al escribir.

In [3]:
silver_plan = (bronze.dropDuplicates(["ride_id"])
               .na.drop(subset=["start_station_id", "end_station_id", "started_at", "ended_at"])
               .filter(duracion.between(1.0, 180.0))
               .filter(en_nyc("start_lat", "start_lng") & en_nyc("end_lat", "end_lng"))
               .withColumn("duracion_min", F.round(duracion, 2))
               .withColumn("fecha", F.to_date("started_at")))
silver_plan.explain()
silver_plan.write.mode("overwrite").partitionBy("fecha").parquet(f"{SALIDA}/silver")

silver = spark.read.parquet(f"{SALIDA}/silver").cache()
n_silver = silver.count()
print(f"Bronze {perfil['filas']:,} -> Silver {n_silver:,} ({perfil['filas'] - n_silver:,} filas descartadas, "
      f"{100 * n_silver / perfil['filas']:.2f} % conservado) | particiones fecha: {silver.select('fecha').distinct().count()}")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [ride_id#0, rideable_type#90, started_at#92, ended_at#94, start_station_name#96, start_station_id#98, end_station_name#100, end_station_id#102, start_lat#104, start_lng#106, end_lat#108, end_lng#110, member_casual#112, round((cast((unix_timestamp(ended_at#94, yyyy-MM-dd HH:mm:ss, Some(America/New_York), true) - unix_timestamp(started_at#92, yyyy-MM-dd HH:mm:ss, Some(America/New_York), true)) as double) / 60.0), 2) AS duracion_min#87, cast(started_at#92 as date) AS fecha#88]
   +- Filter ((((isnotnull(start_lat#104) AND isnotnull(start_lng#106)) AND isnotnull(end_lat#108)) AND isnotnull(end_lng#110)) AND (atleastnnonnulls(4, start_station_id#98, end_station_id#102, started_at#92, ended_at#94) AND (((((((((((cast((unix_timestamp(ended_at#94, yyyy-MM-dd HH:mm:ss, Some(America/New_York), true) - unix_timestamp(started_at#92, yyyy-MM-dd HH:mm:ss, Some(America/New_York), true)) as double) / 60.0) >= 1.0) AND ((cast((unix_time

Bronze 4,993,137 -> Silver 4,971,847 (21,290 filas descartadas, 99.57 % conservado) | particiones fecha: 32


### 4. Criterio espacial: cada estación a su zona

Posición de la estación = promedio de las coordenadas de los viajes que salen o llegan a ella (el GPS de las e-bikes varía unos metros).
Zona = celda de la cuadrícula: `Z{fila}-{columna}` con `fila = floor((lat − 40,50) / 0,02)` y `columna = floor((lon + 74,30) / 0,02)`.
El mapa se guarda en Parquet y en JSON: **el streaming de U2 lo reutiliza**, así batch y streaming hablan de las mismas zonas.

In [4]:
def zona_de(lat, lon):
    fila = F.floor((lat - F.lit(LAT0)) / F.lit(TAM_CELDA)).cast("int")
    col = F.floor((lon - F.lit(LON0)) / F.lit(TAM_CELDA)).cast("int")
    return F.concat(F.lit("Z"), F.lpad(fila.cast("string"), 2, "0"), F.lit("-"), F.lpad(col.cast("string"), 2, "0"))

posiciones = (silver.select(F.col("start_station_id").alias("station_id"), F.col("start_station_name").alias("station_name"),
                            F.col("start_lat").alias("lat"), F.col("start_lng").alias("lon"))
              .unionByName(silver.select(F.col("end_station_id").alias("station_id"), F.col("end_station_name").alias("station_name"),
                                         F.col("end_lat").alias("lat"), F.col("end_lng").alias("lon"))))
estaciones = (posiciones.groupBy("station_id")
              .agg(F.first("station_name", ignorenulls=True).alias("station_name"),
                   F.round(F.avg("lat"), 6).alias("lat"), F.round(F.avg("lon"), 6).alias("lon"))
              .withColumn("zona", zona_de(F.col("lat"), F.col("lon"))))
estaciones.write.mode("overwrite").parquet(f"{SALIDA}/estaciones_zona")
estaciones = spark.read.parquet(f"{SALIDA}/estaciones_zona").cache()

est_pd = estaciones.toPandas()
with open(f"{SALIDA}/estaciones_zona.json", "w", encoding="utf-8") as f:
    json.dump({"tam_celda": TAM_CELDA, "lat0": LAT0, "lon0": LON0,
               "estaciones": dict(zip(est_pd.station_id, est_pd.zona))}, f, ensure_ascii=False)
n_zonas = est_pd.zona.nunique()
print(f"{len(est_pd):,} estaciones en {n_zonas} zonas")

2,418 estaciones en 99 zonas


### 5. Gold: actividad por zona y hora, y ranking de zonas

**Actividad** = salidas + llegadas: una zona puede despachar bicis o recibirlas. Se arma la malla completa zona × hora y las horas sin viajes quedan en **0** (un cero real, no un dato faltante).

In [5]:
mapa = estaciones.select("station_id", "zona")
eventos = (silver.join(mapa.withColumnRenamed("station_id", "start_station_id"), "start_station_id")
                 .select("zona", F.date_trunc("hour", "started_at").alias("hora_ts"), F.lit(1).alias("salida"), F.lit(0).alias("llegada"))
           .unionByName(silver.join(mapa.withColumnRenamed("station_id", "end_station_id"), "end_station_id")
                 .select("zona", F.date_trunc("hour", "ended_at").alias("hora_ts"), F.lit(0).alias("salida"), F.lit(1).alias("llegada"))))
agregado = eventos.groupBy("zona", "hora_ts").agg(F.sum("salida").alias("salidas"), F.sum("llegada").alias("llegadas"))

horas = (agregado.agg(F.min("hora_ts").alias("a"), F.max("hora_ts").alias("b"))
         .select(F.explode(F.sequence("a", "b", F.expr("INTERVAL 1 HOUR"))).alias("hora_ts")))
gold = (horas.crossJoin(mapa.select("zona").distinct())
        .join(agregado, ["zona", "hora_ts"], "left").na.fill(0, ["salidas", "llegadas"])
        .withColumn("actividad", F.col("salidas") + F.col("llegadas"))
        .withColumn("fecha", F.to_date("hora_ts")).withColumn("hora", F.hour("hora_ts"))
        .withColumn("dia_semana", F.dayofweek("hora_ts")).withColumn("finde", F.dayofweek("hora_ts").isin(1, 7).cast("int")))
gold.write.mode("overwrite").partitionBy("fecha").parquet(f"{SALIDA}/gold_zona_hora")
gold = spark.read.parquet(f"{SALIDA}/gold_zona_hora").cache()

centros = estaciones.groupBy("zona").agg(F.count("*").alias("n_estaciones"), F.round(F.avg("lat"), 5).alias("lat"),
                                         F.round(F.avg("lon"), 5).alias("lon"), F.first("station_name").alias("estacion_ejemplo"))
total = gold.agg(F.sum("actividad")).first()[0]
ranking = (gold.groupBy("zona").agg(F.sum("actividad").alias("actividad"), F.sum("salidas").alias("salidas"),
                                    F.sum("llegadas").alias("llegadas"))
           .withColumn("neto", F.col("llegadas") - F.col("salidas"))
           .withColumn("participacion_pct", F.round(100 * F.col("actividad") / F.lit(total), 2))
           .join(centros, "zona").orderBy(F.desc("actividad")))
ranking.write.mode("overwrite").parquet(f"{SALIDA}/gold_ranking_zonas")
rank_pd = ranking.toPandas()
top10_pct = round(float(rank_pd.participacion_pct.head(10).sum()), 2)
print(f"Gold: {gold.count():,} filas zona-hora | {n_zonas} zonas | las 10 zonas top concentran el {top10_pct} % de la actividad")
rank_pd.head(10)[["zona", "estacion_ejemplo", "n_estaciones", "actividad", "participacion_pct", "neto"]]

Gold: 73,953 filas zona-hora | 99 zonas | las 10 zonas top concentran el 52.76 % de la actividad


,zona,estacion_ejemplo,n_estaciones,actividad,participacion_pct,neto
0,Z11-15,E 2 St & Ave B,83,1096249,11.02,933
1,Z12-15,Lexington Ave & E 26 St,65,862202,8.67,-3166
2,Z13-15,W 44 St & 11 Ave,37,515073,5.18,-4187
3,Z12-16,55 Ave & Center Blvd,44,481767,4.84,-3635
4,Z11-14,Grand St & Greene St,37,449875,4.52,1749
5,Z10-14,South St & Broad St,56,419666,4.22,3474
6,Z10-15,Adams St & Prospect St,48,412703,4.15,1253
7,Z13-16,E 65 St & 2 Ave,32,351883,3.54,-625
8,Z09-15,President St & Hoyt St,53,334637,3.37,627
9,Z12-14,8 Ave & W 16 St,21,323265,3.25,691


In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
t = rank_pd.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(t.zona + "  " + t.estacion_ejemplo.str.slice(0, 28), t.actividad, color="#2f6db5")
ax.set_xlabel("actividad (salidas + llegadas), julio 2026"); ax.set_title("U1 · Las 15 zonas con más actividad")
ax.grid(axis="x", alpha=.25); plt.tight_layout(); plt.savefig(f"{SALIDA}/ranking_zonas.png", dpi=120)
print("Gráfico guardado en salidas/u1/ranking_zonas.png")

Gráfico guardado en salidas/u1/ranking_zonas.png


### 6. Modelo de regresión (Spark MLlib): ¿cuánta actividad tendrá una zona en una hora?

- **Objetivo:** `actividad` de la zona en la hora.
- **Features:** hora del día (cíclica), día de la semana, fin de semana, tamaño de la zona (`n_estaciones`), ubicación (`lat`, `lon`), la media histórica de esa zona en esa hora y tipo de día (**solo con train**, para no filtrar información) y la actividad de la misma zona y hora **el día anterior** (`lag_24h`) y **la semana anterior** (`lag_168h`). Uso: planificar mañana con lo que ya pasó hoy.
- **Corte temporal** (el mismo del equipo): primeros 75 % de los días para train, el resto para test.
- Se compara contra un **baseline** (la media histórica sola): el modelo vale lo que mejora sobre él.

In [7]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.sql.window import Window
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd

r = gold.agg(F.min("fecha").alias("d"), F.max("fecha").alias("h")).first()
dias = (r["h"] - r["d"]).days + 1
CORTE = r["d"] + dt.timedelta(days=int(dias * 0.75))
print(f"Rango {r['d']} -> {r['h']} ({dias} días) | corte train/test: {CORTE}")

w = Window.partitionBy("zona").orderBy("hora_ts")
media = (gold.filter(F.col("fecha") < F.lit(CORTE)).groupBy("zona", "finde", "hora")
         .agg(F.avg("actividad").alias("media_zona_hora")))
datos = (gold.join(centros.select("zona", "n_estaciones", "lat", "lon"), "zona")
         .join(media, ["zona", "finde", "hora"], "left").na.fill(0.0, ["media_zona_hora"])
         .withColumn("hora_sin", F.sin(2 * 3.141592653589793 * F.col("hora") / 24))
         .withColumn("hora_cos", F.cos(2 * 3.141592653589793 * F.col("hora") / 24))
         .withColumn("lag_24h", F.lag("actividad", 24).over(w)).withColumn("lag_168h", F.lag("actividad", 168).over(w))
         .withColumn("label", F.col("actividad").cast("double"))
         .na.drop(subset=["lag_24h", "lag_168h"]))          # la primera semana no tiene lag de 7 días
train = datos.filter(F.col("fecha") < F.lit(CORTE)).cache()
test = datos.filter(F.col("fecha") >= F.lit(CORTE)).cache()
print(f"Train {train.count():,} filas | Test {test.count():,} filas")

FEATURES = ["hora_sin", "hora_cos", "dia_semana_vec", "finde", "n_estaciones", "lat", "lon", "media_zona_hora", "lag_24h", "lag_168h"]
def pipeline(modelo):
    return Pipeline(stages=[OneHotEncoder(inputCols=["dia_semana"], outputCols=["dia_semana_vec"]),
                            VectorAssembler(inputCols=FEATURES, outputCol="features"), modelo])

ev = {m: RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName=m) for m in ("rmse", "mae", "r2")}
def evaluar(pred, nombre):
    return {"modelo": nombre, **{m.upper(): round(e.evaluate(pred), 4) for m, e in ev.items()}}

resultados = [evaluar(test.withColumn("prediction", F.col("media_zona_hora")), "Baseline media histórica")]
candidatos = {"LinearRegression": LinearRegression(labelCol="label", regParam=0.1),
              "GBTRegressor": GBTRegressor(labelCol="label", maxIter=60, maxDepth=5, seed=42)}
modelos = {}
for nombre, est in candidatos.items():
    modelos[nombre] = pipeline(est).fit(train)
    resultados.append(evaluar(modelos[nombre].transform(test), nombre))
tabla = pd.DataFrame(resultados)
tabla["mejora_RMSE_vs_baseline_%"] = (100 * (tabla.RMSE.iloc[0] - tabla.RMSE) / tabla.RMSE.iloc[0]).round(2)
tabla

Rango 2026-06-30 -> 2026-07-31 (32 días) | corte train/test: 2026-07-24


Train 38,313 filas | Test 19,008 filas


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


,modelo,RMSE,MAE,R2,mejora_RMSE_vs_baseline_%
0,Baseline media histórica,78.9370,28.8327,0.9413,0.00
1,LinearRegression,75.7215,30.5797,0.9460,4.07
2,GBTRegressor,71.2870,25.3602,0.9522,9.69


### 7. Selección del modelo, guardado y zonas con alta actividad

In [8]:
ganador = tabla[tabla.modelo.isin(list(modelos))].sort_values("RMSE").iloc[0]
modelos[ganador.modelo].write().overwrite().save(f"{SALIDA}/modelo_actividad_zona")
recargado = PipelineModel.load(f"{SALIDA}/modelo_actividad_zona")
pred = recargado.transform(test).cache()
print(f"Ganador: {ganador.modelo} | RMSE {ganador.RMSE} | MAE {ganador.MAE} | R² {ganador.R2} "
      f"| recargado coincide: {round(ev['rmse'].evaluate(pred), 4) == ganador.RMSE}")

# "Alta actividad" = la hora de la zona supera el percentil 75 de las horas con actividad en train
UMBRAL = float(train.filter("actividad > 0").approxQuantile("actividad", [0.75], 0.001)[0])
c = (pred.withColumn("real_alta", (F.col("actividad") >= UMBRAL).cast("int"))
         .withColumn("pred_alta", (F.col("prediction") >= UMBRAL).cast("int"))
         .agg(F.sum(F.col("real_alta") * F.col("pred_alta")).alias("tp"), F.sum((1 - F.col("real_alta")) * F.col("pred_alta")).alias("fp"),
              F.sum(F.col("real_alta") * (1 - F.col("pred_alta"))).alias("fn"), F.count("*").alias("n"),
              F.sum((F.col("real_alta") == F.col("pred_alta")).cast("int")).alias("ok")).first())
precision = c.tp / (c.tp + c.fp) if c.tp + c.fp else 0.0
recall = c.tp / (c.tp + c.fn) if c.tp + c.fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
print(f"Umbral de alta actividad: {UMBRAL:.0f} viajes/hora por zona | accuracy {c.ok / c.n:.4f} | "
      f"precision {precision:.4f} | recall {recall:.4f} | F1 {f1:.4f}")

alta = (pred.groupBy("zona").agg(F.round(100 * F.avg((F.col("prediction") >= UMBRAL).cast("int")), 1).alias("pct_horas_alta_predichas"),
                                 F.round(F.avg("prediction"), 1).alias("actividad_media_predicha"))
        .join(centros.select("zona", "estacion_ejemplo"), "zona").orderBy(F.desc("pct_horas_alta_predichas"), F.desc("actividad_media_predicha")))
alta.write.mode("overwrite").parquet(f"{SALIDA}/gold_zonas_alta_actividad")
alta_pd = alta.toPandas()
alta_pd.head(10)

Ganador: GBTRegressor | RMSE 71.287 | MAE 25.3602 | R² 0.9522 | recargado coincide: True


Umbral de alta actividad: 152 viajes/hora por zona | accuracy 0.9743 | precision 0.9366 | recall 0.9459 | F1 0.9413


,zona,pct_horas_alta_predichas,actividad_media_predicha,estacion_ejemplo
0,Z11-15,91.7,1582.2,E 2 St & Ave B
1,Z12-15,89.6,1251.3,Lexington Ave & E 26 St
2,Z13-15,84.4,748.5,W 44 St & 11 Ave
3,Z12-16,82.8,710.5,55 Ave & Center Blvd
4,Z10-15,80.2,600.5,Adams St & Prospect St
5,Z11-14,79.7,664.1,Grand St & Greene St
6,Z10-17,78.6,423.6,Union Ave & Wallabout St
7,Z10-14,77.6,617.0,South St & Broad St
8,Z09-16,75.5,403.5,Carlton Ave & Dean St
9,Z09-15,75.0,490.7,President St & Hoyt St


In [9]:
metricas = {"bronze_filas": perfil["filas"], "silver_filas": n_silver, "perfil_calidad": perfil,
            "estaciones": len(est_pd), "zonas": int(n_zonas), "top10_participacion_pct": top10_pct,
            "corte": str(CORTE), "resultados": resultados, "ganador": ganador.modelo,
            "umbral_alta_actividad": UMBRAL, "alta_actividad": {"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4),
                                                                "accuracy": round(c.ok / c.n, 4)},
            "top5_zonas": rank_pd.head(5)[["zona", "estacion_ejemplo", "actividad", "participacion_pct"]].to_dict(orient="records")}
with open(f"{SALIDA}/metricas_u1.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2, default=str)
print("metricas_u1.json guardado")

metricas_u1.json guardado


### 8. Respuesta a la pregunta U1

1. **¿Qué zonas concentran la actividad?** El ranking Gold (sección 5): pocas zonas de Manhattan concentran gran parte de los viajes. El porcentaje exacto y el top 10 están arriba y en `gold_ranking_zonas`.
2. **¿Cuáles tienen mayor probabilidad de alta actividad?** Las de la tabla de la sección 7: el modelo predice que superarán el umbral (P75) en la mayor parte de sus horas.
3. **Por qué se puede creer:** corte temporal sin fuga, comparación contra un baseline y modelo guardado y recargado con el mismo resultado.
4. **Puente a U2:** `silver/` y `estaciones_zona` son la entrada de `2_U2_streaming/U2_01_modelo_series_tiempo_zonas.ipynb`.

---
# 02 · UNIDAD II — PREDICCIÓN EN TIEMPO REAL (STREAMING)

**¿Qué zonas concentrarán más actividad en la siguiente ventana de 15 minutos?**

Se cierra la sesión de U1 para liberar su memoria (la capa batch ya dejó sus resultados en Parquet); la celda de sesión de la Parte A abre una nueva.

In [10]:
spark.stop()
import gc; gc.collect()
print('Sesión de U1 cerrada: memoria liberada')

Sesión de U1 cerrada: memoria liberada


## Parte A · Modelo de series de tiempo por zona (antes notebook U2_01)

### U2 · 01 — Modelo de series de tiempo por zona (se entrena en batch, se usa en streaming)

**Integrante:** Cristhian Chuquitarqui Chura — Data Geniuses · **Proyecto Sello:** CitiBike Intelligent Mobility

**Pregunta U2 (Brief S2):** *¿Qué zonas presentan mayor concentración actual y cuáles podrían concentrar más actividad en la siguiente ventana?*

**Evolución desde U1.** En U1 la pregunta era qué zonas concentran la actividad histórica y cuáles tienen mayor probabilidad de alta actividad. En U2 se pronostica **cuánta actividad (salidas + llegadas) tendrá cada zona en los próximos 15 minutos**, y se muestra en vivo junto con el estado real de los docks por zona.

| Decisión | U2 (este notebook) |
|---|---|
| Zona | **Cuadrante** de una cuadrícula fija de 0,02° × 0,02° (~2,2 km × 1,7 km) sobre las coordenadas de cada estación: criterio espacial reproducible |
| Actividad | salidas + llegadas de la zona por ventana de 15 min |
| Zonas modeladas | las **15** con más actividad en el período de entrenamiento |
| Corte train/test | el mismo de U1 del equipo: primeros 75 % de los días (2026-07-24) |
| Entrada | **la salida de U1**: `salidas/u1/silver` y `salidas/u1/estaciones_zona` (mismas zonas en batch y streaming) |

#### 1. SparkSession

Zona horaria `America/New_York`: los CSV vienen en hora local de NYC y la hora del día es una variable del modelo. El notebook 02 usa la misma.

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("U2_Espacial_01_Zonas_CristhianChuquitarqui")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 4.2.0


In [12]:
import json, os, shutil

U1 = "../salidas/u1"                # salida del notebook 1_U1_batch/U1_etl_zonas_alta_actividad.ipynb
ARTIFACTS = "../salidas/u2"         # lo que usa el streaming (notebook U2_02)
VENTANA_MIN = 15
TOP_K = 15                          # zonas que se modelan
os.makedirs(ARTIFACTS, exist_ok=True)
assert os.path.isdir(f"{U1}/silver"), "Primero corre el notebook de U1 (1_U1_batch)"

#### 2. Entrada: Silver de U1 (Lambda: la capa batch alimenta a la de velocidad)

No se repite la limpieza: se lee el Parquet Silver que dejó U1 (mismas reglas de calidad del equipo).

In [13]:
silver = spark.read.parquet(f"{U1}/silver").cache()
print(f"Silver de U1: {silver.count():,} viajes")

Silver de U1: 4,971,847 viajes


#### 3. Zonas: el mismo mapa estación → zona de U1

Se reutiliza la cuadrícula de 0,02° de U1. El JSON se copia a `salidas/u2/` porque el streaming (U2_02) lo usa para asignar cada evento de Kafka a **la misma** zona.

In [14]:
estaciones = spark.read.parquet(f"{U1}/estaciones_zona").cache()
shutil.copy(f"{U1}/estaciones_zona.json", f"{ARTIFACTS}/estaciones_zona.json")
print(f"Estaciones: {estaciones.count():,} | zonas distintas: {estaciones.select('zona').distinct().count()}")

Estaciones: 2,418 | zonas distintas: 99


#### 4. Corte temporal (el mismo de U1) y zonas top-K

In [15]:
import datetime as dt

rango = silver.agg(F.min(F.to_date("started_at")).alias("desde"), F.max(F.to_date("started_at")).alias("hasta")).first()
desde, hasta = rango["desde"], rango["hasta"]
dias = (hasta - desde).days + 1
CORTE = desde + dt.timedelta(days=int(dias * 0.75))
print(f"Rango: {desde} -> {hasta} ({dias} días) | corte train/test: {CORTE}")

mapa = estaciones.select("station_id", "zona")
viajes_z = (silver
    .join(mapa.withColumnRenamed("station_id", "start_station_id").withColumnRenamed("zona", "zona_origen"), "start_station_id")
    .join(mapa.withColumnRenamed("station_id", "end_station_id").withColumnRenamed("zona", "zona_destino"), "end_station_id")
    .cache())

train_v = viajes_z.filter(F.to_date("started_at") < F.lit(CORTE))
actividad_train = (train_v.select(F.col("zona_origen").alias("zona")).unionByName(train_v.select(F.col("zona_destino").alias("zona")))
                   .groupBy("zona").agg(F.count("*").alias("actividad_train")))
centros = estaciones.groupBy("zona").agg(F.round(F.avg("lat"), 5).alias("lat"), F.round(F.avg("lon"), 5).alias("lon"),
                                          F.count("*").alias("n_estaciones"),
                                          F.first("station_name").alias("estacion_ejemplo"))
top = (actividad_train.join(centros, "zona").orderBy(F.desc("actividad_train")).limit(TOP_K))
top_pd = top.toPandas()
total_train = actividad_train.agg(F.sum("actividad_train")).first()[0]
with open(f"{ARTIFACTS}/zonas_top.json", "w", encoding="utf-8") as f:
    json.dump({"criterio": f"top {TOP_K} zonas por actividad (salidas+llegadas) en train (< {CORTE})",
               "participacion_pct": round(100 * float(top_pd["actividad_train"].sum()) / total_train, 2),
               "zonas": top_pd.to_dict(orient="records")}, f, ensure_ascii=False, indent=2, default=str)
print(f"Las {TOP_K} zonas top concentran el {100 * top_pd['actividad_train'].sum() / total_train:.2f} % de la actividad de train")
top_pd

Rango: 2026-06-30 -> 2026-07-31 (32 días) | corte train/test: 2026-07-24


Las 15 zonas top concentran el 65.57 % de la actividad de train


,zona,actividad_train,lat,lon,n_estaciones,estacion_ejemplo
0,Z11-15,791068,40.72960,-73.98961,83,E 2 St & Ave B
1,Z12-15,626952,40.74998,-73.99012,65,Lexington Ave & E 26 St
2,Z13-15,374516,40.76694,-73.98734,37,W 44 St & 11 Ave
3,Z12-16,350178,40.75272,-73.97260,44,55 Ave & Center Blvd
4,Z11-14,323222,40.73004,-74.00595,37,Grand St & Greene St
5,Z10-14,301844,40.71084,-74.00912,56,South St & Broad St
6,Z10-15,297992,40.71106,-73.99006,48,Adams St & Prospect St
7,Z13-16,253086,40.76872,-73.97001,32,E 65 St & 2 Ave
8,Z09-15,242582,40.68943,-73.99018,53,President St & Hoyt St
9,Z12-14,234549,40.74664,-74.00391,21,8 Ave & W 16 St


#### 5. Serie zona × ventana de 15 min, con ceros explícitos

Una ventana sin viajes es un **0** real. Se arma la malla completa (ventanas × zonas top) y se completa con 0, para que los lags no salten sobre los huecos.

In [16]:
ids_top = [r["zona"] for r in top.select("zona").collect()]
ventana = f"{VENTANA_MIN} minutes"
salidas = (viajes_z.filter(F.col("zona_origen").isin(ids_top))
           .groupBy(F.col("zona_origen").alias("zona"), F.window("started_at", ventana).start.alias("slot"))
           .agg(F.count("*").alias("salidas")))
llegadas = (viajes_z.filter(F.col("zona_destino").isin(ids_top))
            .groupBy(F.col("zona_destino").alias("zona"), F.window("ended_at", ventana).start.alias("slot"))
            .agg(F.count("*").alias("llegadas")))

malla = (salidas.agg(F.min("slot").alias("a"), F.max("slot").alias("b"))
         .select(F.explode(F.sequence("a", "b", F.expr(f"INTERVAL {VENTANA_MIN} MINUTES"))).alias("slot"))
         .crossJoin(top.select("zona")))
serie = (malla.join(salidas, ["zona", "slot"], "left").join(llegadas, ["zona", "slot"], "left")
              .na.fill(0, ["salidas", "llegadas"])
              .withColumn("actividad", F.col("salidas") + F.col("llegadas"))
              .cache())
print(f"Filas de la serie: {serie.count():,}")
serie.orderBy("zona", "slot").show(5)

Filas de la serie: 44,775
+------+-------------------+-------+--------+---------+
|  zona|               slot|salidas|llegadas|actividad|
+------+-------------------+-------+--------+---------+
|Z08-16|2026-06-30 21:45:00|      0|       0|        0|
|Z08-16|2026-06-30 22:00:00|      0|       0|        0|
|Z08-16|2026-06-30 22:15:00|      0|       0|        0|
|Z08-16|2026-06-30 22:30:00|      0|       0|        0|
|Z08-16|2026-06-30 22:45:00|      0|       0|        0|
+------+-------------------+-------+--------+---------+
only showing top 5 rows


#### 6. Features y objetivo

Cada fila es el cierre de la ventana **t**; se predice la actividad de **t+1**.

| Feature | Qué es |
|---|---|
| `a_t` … `a_t3` | actividad en t, t-1, t-2, t-3 (la última hora) |
| `neto_t` | llegadas − salidas en t (si la zona está recibiendo o despachando bicis) |
| `hora_sin`, `hora_cos` | hora del día de la ventana objetivo, cíclica |
| `finde_obj` | la ventana objetivo cae en fin de semana |
| `media_slot_obj` | media histórica de actividad de **esa zona** en esa franja y tipo de día (solo train) |

In [17]:
w = Window.partitionBy("zona").orderBy("slot")
base = (serie
        .withColumn("a_t", F.col("actividad"))
        .withColumn("a_t1", F.lag("actividad", 1).over(w))
        .withColumn("a_t2", F.lag("actividad", 2).over(w))
        .withColumn("a_t3", F.lag("actividad", 3).over(w))
        .withColumn("neto_t", F.col("llegadas") - F.col("salidas"))
        .withColumn("label", F.lead("actividad", 1).over(w))
        .withColumn("slot_obj", F.col("slot") + F.expr(f"INTERVAL {VENTANA_MIN} MINUTES"))
        .withColumn("hora_obj", F.hour("slot_obj") + F.minute("slot_obj") / 60.0)
        .withColumn("finde_obj", F.dayofweek("slot_obj").isin(1, 7).cast("int"))
        .withColumn("hora_slot_obj", F.hour("slot_obj"))
        .na.drop(subset=["a_t1", "a_t2", "a_t3", "label"]))

media_slot = (serie.filter(F.to_date("slot") < F.lit(CORTE))
              .withColumn("finde", F.dayofweek("slot").isin(1, 7).cast("int"))
              .groupBy("zona", "finde", F.hour("slot").alias("hora"))
              .agg(F.round(F.avg("actividad"), 4).alias("media_slot")))

dataset = (base.join(media_slot.withColumnRenamed("finde", "finde_obj").withColumnRenamed("hora", "hora_slot_obj")
                     .withColumnRenamed("media_slot", "media_slot_obj"), ["zona", "finde_obj", "hora_slot_obj"], "left")
               .na.fill(0.0, ["media_slot_obj"])
               .withColumn("hora_sin", F.sin(2 * 3.141592653589793 * F.col("hora_obj") / 24))
               .withColumn("hora_cos", F.cos(2 * 3.141592653589793 * F.col("hora_obj") / 24))
               .withColumn("label", F.col("label").cast("double")))

FEATURES = ["a_t", "a_t1", "a_t2", "a_t3", "neto_t", "hora_sin", "hora_cos", "finde_obj", "media_slot_obj"]
train = dataset.filter(F.to_date("slot_obj") < F.lit(CORTE)).cache()
test = dataset.filter(F.to_date("slot_obj") >= F.lit(CORTE)).cache()
lim = (train.agg(F.max("slot_obj").alias("m")).crossJoin(test.agg(F.min("slot_obj").alias("n")))
       .select((F.col("m") < F.col("n")).alias("ok"), F.date_format("m", "yyyy-MM-dd HH:mm").alias("m"),
               F.date_format("n", "yyyy-MM-dd HH:mm").alias("n")).first())
assert lim["ok"], "El split no respeta el orden temporal"
print(f"Train: {train.count():,} filas (hasta {lim['m']}) | Test: {test.count():,} filas (desde {lim['n']}, hora NYC)")

Train: 33,195 filas (hasta 2026-07-23 23:45) | Test: 11,520 filas (desde 2026-07-24 00:00, hora NYC)


#### 7. Baselines y modelos (Spark MLlib)

In [18]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, GBTRegressor
import pandas as pd

ev = {m: RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName=m) for m in ("rmse", "mae", "r2")}
def evaluar(pred, nombre):
    r = {"modelo": nombre, **{m.upper(): round(e.evaluate(pred), 4) for m, e in ev.items()}}
    print(f"{nombre:32s} RMSE={r['RMSE']:.4f}  MAE={r['MAE']:.4f}  R2={r['R2']:.4f}")
    return r

resultados = [evaluar(test.withColumn("prediction", F.col("a_t").cast("double")), "Baseline persistencia"),
              evaluar(test.withColumn("prediction", F.col("media_slot_obj")), "Baseline media histórica")]
pipeline = lambda est: Pipeline(stages=[VectorAssembler(inputCols=FEATURES, outputCol="features"), est])
candidatos = {"LinearRegression": pipeline(LinearRegression(labelCol="label")),
              "GBTRegressor (60 árboles)": pipeline(GBTRegressor(labelCol="label", maxIter=60, maxDepth=5, seed=42))}
modelos = {}
for nombre, p in candidatos.items():
    modelos[nombre] = p.fit(train)
    resultados.append(evaluar(modelos[nombre].transform(test), nombre))

tabla = pd.DataFrame(resultados)
rmse_pers = tabla.loc[tabla.modelo == "Baseline persistencia", "RMSE"].iloc[0]
tabla["mejora_RMSE_vs_persistencia_%"] = (100 * (rmse_pers - tabla["RMSE"]) / rmse_pers).round(2)
tabla

Baseline persistencia            RMSE=30.7977  MAE=20.1174  R2=0.9605


Baseline media histórica         RMSE=54.7160  MAE=34.3748  R2=0.8753


LinearRegression                 RMSE=27.7417  MAE=18.2092  R2=0.9680


GBTRegressor (60 árboles)        RMSE=29.2785  MAE=17.7542  R2=0.9643


,modelo,RMSE,MAE,R2,mejora_RMSE_vs_persistencia_%
0,Baseline persistencia,30.7977,20.1174,0.9605,0.00
1,Baseline media histórica,54.7160,34.3748,0.8753,-77.66
2,LinearRegression,27.7417,18.2092,0.9680,9.92
3,GBTRegressor (60 árboles),29.2785,17.7542,0.9643,4.93


#### 8. Selección, persistencia y verificación

In [19]:
ganador = tabla[tabla.modelo.isin(list(modelos))].sort_values("RMSE").iloc[0]
print("Modelo ganador:", ganador["modelo"], "| RMSE", ganador["RMSE"], "| MAE", ganador["MAE"], "| R2", ganador["R2"])
RUTA_MODELO = f"{ARTIFACTS}/models/modelo_zonas_ts"
modelos[ganador["modelo"]].write().overwrite().save(RUTA_MODELO)
media_slot.write.mode("overwrite").parquet(f"{ARTIFACTS}/media_slot")
recargado = PipelineModel.load(RUTA_MODELO)
rmse_r = ev["rmse"].evaluate(recargado.transform(test))
print(f"RMSE recargado: {rmse_r:.4f} (esperado {ganador['RMSE']}) -> coincide: {round(rmse_r, 4) == ganador['RMSE']}")
with open(f"{ARTIFACTS}/metricas_modelo.json", "w", encoding="utf-8") as f:
    json.dump({"ventana_min": VENTANA_MIN, "top_k": TOP_K, "corte": str(CORTE), "features": FEATURES,
               "ganador": ganador["modelo"], "resultados": resultados}, f, ensure_ascii=False, indent=2)

Modelo ganador: LinearRegression | RMSE 27.7417 | MAE 18.2092 | R2 0.968


RMSE recargado: 27.7417 (esperado 27.7417) -> coincide: True


#### 9. Error por zona

In [20]:
pred_test = recargado.transform(test)
(pred_test.groupBy("zona").agg(F.round(F.avg(F.abs(F.col("prediction") - F.col("label"))), 3).alias("MAE"),
                               F.round(F.avg("label"), 2).alias("actividad_media"))
 .withColumn("MAE_relativo", F.round(F.col("MAE") / F.col("actividad_media"), 3))
 .join(top.select("zona", "n_estaciones", "estacion_ejemplo"), "zona")
 .orderBy(F.desc("MAE")).show(TOP_K, truncate=False))

+------+------+---------------+------------+------------+------------------------+
|zona  |MAE   |actividad_media|MAE_relativo|n_estaciones|estacion_ejemplo        |
+------+------+---------------+------------+------------+------------------------+
|Z11-15|36.698|397.51         |0.092       |83          |E 2 St & Ave B          |
|Z12-15|30.045|306.38         |0.098       |65          |Lexington Ave & E 26 St |
|Z12-16|22.373|171.39         |0.131       |44          |55 Ave & Center Blvd    |
|Z11-14|19.585|164.96         |0.119       |37          |Grand St & Greene St    |
|Z10-14|19.412|153.45         |0.127       |56          |South St & Broad St     |
|Z13-15|19.061|183.06         |0.104       |37          |W 44 St & 11 Ave        |
|Z10-15|17.195|149.4          |0.115       |48          |Adams St & Prospect St  |
|Z13-16|16.803|128.68         |0.131       |32          |E 65 St & 2 Ave         |
|Z12-14|15.098|115.54         |0.131       |21          |8 Ave & W 16 St         |
|Z09

#### 10. Publicar las métricas del modelo en Grafana (Pushgateway)

In [21]:
import urllib.request
lineas = []
for r in resultados:
    for m in ("RMSE", "MAE", "R2"):
        lineas.append(f'lambda26_u2esp_modelo_{m.lower()}_test{{modelo="{r["modelo"]}"}} {r[m]}')
lineas.append(f'lambda26_u2esp_modelo_ganador_info{{modelo="{ganador["modelo"]}"}} 1')
try:
    req = urllib.request.Request("http://pushgateway:9091/metrics/job/lambda26_u2_espacial_modelo/instance/batch",
                                 data=("\n".join(lineas) + "\n").encode(), method="PUT")
    urllib.request.urlopen(req, timeout=5).close()
    print("Pushgateway: métricas del modelo publicadas")
except Exception as ex:
    print("Pushgateway no disponible (no es error del modelo):", ex)

Pushgateway: métricas del modelo publicadas


#### 11. Real vs. predicho en test, para la zona con más actividad

In [22]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
    import matplotlib.pyplot as plt

z = ids_top[0]
g = (pred_test.filter(F.col("zona") == z).orderBy("slot_obj")
     .select("slot_obj", "label", "prediction", "media_slot_obj").limit(4 * 24 * 3).toPandas())
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(g.slot_obj, g.label, label="Real", color="#4C78A8")
ax.plot(g.slot_obj, g.prediction, label=f"Predicho ({ganador['modelo']})", color="#C00000")
ax.plot(g.slot_obj, g.media_slot_obj, label="Media histórica", color="#9AA7B0", linestyle="--")
ax.set_title(f"Actividad por ventana de {VENTANA_MIN} min — zona {z} — primeros 3 días de test")
ax.set_ylabel("salidas + llegadas"); ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); plt.savefig(f"{ARTIFACTS}/real_vs_predicho.png", dpi=120); plt.show()

#### 12. Síntesis para la sustentación

1. **Criterio espacial reproducible:** cuadrantes fijos de 0,02°. Cada estación tiene una sola zona, guardada en un archivo que el streaming reutiliza.
2. **Actividad = salidas + llegadas:** una zona puede despachar bicis o recibirlas. `neto_t` dice cuál de las dos pasa.
3. **Sin fuga de información:** el corte es el de U1; `media_slot` y el top-K salen solo de train.
4. **Contra baselines:** el modelo se justifica por lo que mejora frente a la persistencia y a la media histórica.
5. **Escalable:** un modelo global para todas las zonas. Cuadrantes más finos dan más zonas, no más modelos.

## Parte B · Streaming por zona, inferencia en vivo y observabilidad (antes notebook U2_02)

Las celdas marcadas con la etiqueta `u2_streaming` son las que `scripts/streaming_en_segundo_plano.py` vuelve a ejecutar para dejar el streaming corriendo sin Jupyter.

In [23]:
spark.stop()
gc.collect()
print('Sesión de la Parte A cerrada: memoria liberada')

Sesión de la Parte A cerrada: memoria liberada


### U2 · 02 — Streaming por zona, inferencia en vivo y observabilidad

**Integrante:** Cristhian Chuquitarqui Chura — Data Geniuses · **Proyecto Sello:** CitiBike Intelligent Mobility

Tres consultas de Spark Structured Streaming y un monitor, publicados en Grafana (tablero **"U2 · Espacial — Cristhian"**):

| Consulta | Topic | Qué hace | Criterio U2 |
|---|---|---|---|
| `u2z_actividad` | `citibike-rides` (eventos empresariales) | Asigna cada viaje a su zona; salidas, llegadas y actividad por zona en ventanas de 15 min, con watermark y checkpoint; pronóstico de la ventana t+1 | 1, 3, 6, 7 |
| `u2z_estado_iot` | `citibike-estaciones` (sensores de docks, todas las estaciones) | Deduplica reportes del sensor y agrega por zona: bicis, docks, ocupación, estaciones vacías y llenas | 2, 6 |
| `u2z_calidad` | ambos | Válidos e inválidos, latencia de ingesta y bytes por evento | 4, 5 |
| monitor | `lastProgress` | Throughput, duración del micro-lote, retraso de offsets, filas descartadas | 4 |

**Antes:** `docker compose up -d --build --wait` (levanta Kafka, productores y Grafana) y los notebooks U1 y U2_01 ejecutados (ver `README.md`).

In [24]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("U2_Espacial_02_Streaming_CristhianChuquitarqui")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 4.2.0


#### 1. Parámetros y artefactos del notebook 01

In [25]:
import json, math, os, shutil, threading, time, urllib.request
from collections import defaultdict, deque
from datetime import datetime
from zoneinfo import ZoneInfo
from pyspark.ml import PipelineModel

KAFKA = "kafka:9092"
TOPIC_RIDES = "citibike-rides"
TOPIC_ESTACIONES = "citibike-estaciones"
PUSHGATEWAY = "http://pushgateway:9091"
ARTIFACTS = "../salidas/u2"            # salida de U2_01
STREAM_OUT = f"{ARTIFACTS}/stream"
CHECKPOINTS = f"{ARTIFACTS}/checkpoints"
VENTANA_MIN = 15
WATERMARK_ACT = "10 minutes"
WATERMARK_IOT = "15 minutes"
SPEEDUP = 60
NY = ZoneInfo("America/New_York")

with open(f"{ARTIFACTS}/zonas_top.json", encoding="utf-8") as f:
    TOP = json.load(f)["zonas"]
IDS_TOP = [z["zona"] for z in TOP]
with open(f"{ARTIFACTS}/estaciones_zona.json", encoding="utf-8") as f:
    GRID = json.load(f)
EST_ZONA = GRID["estaciones"]
TAM, LAT0, LON0 = GRID["tam_celda"], GRID["lat0"], GRID["lon0"]
with open(f"{ARTIFACTS}/metricas_modelo.json", encoding="utf-8") as f:
    META = json.load(f)
FEATURES = META["features"]
assert META["ventana_min"] == VENTANA_MIN

modelo = PipelineModel.load(f"{ARTIFACTS}/models/modelo_zonas_ts")
MEDIA_SLOT = {(r["zona"], r["finde"], r["hora"]): r["media_slot"] for r in spark.read.parquet(f"{ARTIFACTS}/media_slot").collect()}
mapa_df = spark.createDataFrame(list(EST_ZONA.items()), "stationId string, zona string")   # tabla estática para el join
print(f"Modelo: {META['ganador']} | {len(IDS_TOP)} zonas top | {len(EST_ZONA)} estaciones con zona")

Modelo: LinearRegression | 15 zonas top | 2418 estaciones con zona


#### 2. Publicación en Pushgateway (y re-publicación de las métricas del modelo)

In [26]:
JOB = "lambda26_u2_espacial"

def _esc(v):
    return str(v).replace("\\", "\\\\").replace('"', '\\"')

def metrica(_metrica, _valor, **labels):
    if _valor is None:
        return None
    lab = ",".join(f'{k}="{_esc(v)}"' for k, v in labels.items())
    return f"lambda26_u2esp_{_metrica}{{{lab}}} {float(_valor)}" if lab else f"lambda26_u2esp_{_metrica} {float(_valor)}"

def push(instancia, lineas, job=JOB):
    cuerpo = "\n".join(l for l in lineas if l) + "\n"
    req = urllib.request.Request(f"{PUSHGATEWAY}/metrics/job/{job}/instance/{instancia}", data=cuerpo.encode(), method="PUT")
    try:
        urllib.request.urlopen(req, timeout=5).close()
    except Exception as ex:
        print(f"[push {instancia}] Pushgateway no disponible: {ex}")

# El Pushgateway pierde todo al reiniciarse: se re-publican las métricas de test del notebook 01
push("batch", [metrica(f"modelo_{m.lower()}_test", r[m], modelo=r["modelo"]) for r in META["resultados"] for m in ("RMSE", "MAE", "R2")]
     + [metrica("modelo_ganador_info", 1, modelo=META["ganador"])], job="lambda26_u2_espacial_modelo")
print("Métricas del modelo publicadas")

Métricas del modelo publicadas


#### 3. Lectura de Kafka, contrato y zona de cada evento

- **Viajes:** la zona sale del mapa estación → zona del notebook 01 (un *join* stream-estático). Un viaje cuya estación no está en el mapa queda con `zona = null` y se cuenta aparte.
- **Docks (IoT):** primero se busca en el mismo mapa. Si la estación es nueva, se calcula su zona con la misma cuadrícula, usando el `lat`/`lon` que trae el feed.

In [27]:
from pyspark.sql.types import StructType, StringType, LongType, IntegerType, BooleanType, DoubleType

def leer_topic(topics, starting="latest"):
    return (spark.readStream.format("kafka")
            .option("kafka.bootstrap.servers", KAFKA)
            .option("subscribe", topics)
            .option("startingOffsets", starting)
            .option("failOnDataLoss", "false")
            .option("maxOffsetsPerTrigger", 20000)
            .load())

esquema_viaje = (StructType().add("tipoEvento", StringType()).add("stationId", StringType()).add("stationName", StringType())
    .add("rideId", StringType()).add("rideableType", StringType()).add("memberCasual", StringType())
    .add("origen", StringType()).add("timestamp", LongType()).add("publishedAt", LongType()))
esquema_estado = (StructType().add("tipoEvento", StringType()).add("stationId", StringType()).add("stationName", StringType())
    .add("capacidad", IntegerType()).add("lat", DoubleType()).add("lon", DoubleType())
    .add("bicisDisponibles", IntegerType()).add("ebikesDisponibles", IntegerType()).add("docksDisponibles", IntegerType())
    .add("enServicio", BooleanType()).add("origen", StringType()).add("timestamp", LongType()).add("publishedAt", LongType()))

VALIDO_VIAJE = (F.col("tipoEvento").isin("viaje.salida", "viaje.llegada") & F.col("stationId").isNotNull()
                & (F.col("timestamp") > 0) & F.col("publishedAt").isNotNull())
VALIDO_ESTADO = ((F.col("tipoEvento") == "estacion.estado") & F.col("stationId").isNotNull()
                 & F.col("bicisDisponibles").between(0, 150) & F.col("docksDisponibles").between(0, 150)
                 & (F.col("timestamp") > 0) & F.col("publishedAt").isNotNull())

def zona_grid(lat, lon):
    fila = F.floor((lat - F.lit(LAT0)) / F.lit(TAM)).cast("int")
    col = F.floor((lon - F.lit(LON0)) / F.lit(TAM)).cast("int")
    return F.concat(F.lit("Z"), F.lpad(fila.cast("string"), 2, "0"), F.lit("-"), F.lpad(col.cast("string"), 2, "0"))

def parsear(raw, esquema, regla):
    return (raw.select(F.col("value").cast("string").alias("json"))
               .select(F.from_json("json", esquema).alias("e")).select("e.*")
               .withColumn("eventTime", F.timestamp_millis("timestamp"))
               .withColumn("publishedTime", F.timestamp_millis("publishedAt"))
               .withColumn("isValid", F.coalesce(regla, F.lit(False))))

viajes = parsear(leer_topic(TOPIC_RIDES), esquema_viaje, VALIDO_VIAJE).join(mapa_df, "stationId", "left")
estados = (parsear(leer_topic(TOPIC_ESTACIONES), esquema_estado, VALIDO_ESTADO)
           .join(mapa_df.withColumnRenamed("zona", "zonaMapa"), "stationId", "left")
           .withColumn("zona", F.coalesce("zonaMapa", zona_grid(F.col("lat"), F.col("lon")))).drop("zonaMapa"))

#### 4. Consulta `u2z_actividad`: ventanas + watermark + pronóstico t+1 por zona

Ventana tumbling de 15 min sobre el **tiempo del evento**, con watermark de 10 min y checkpoint. En `foreachBatch`, por cada ventana cerrada:

1. Se completa con 0 las zonas top sin viajes.
2. Se compara con la predicción hecha antes para esa ventana (MAE en producción).
3. Se actualiza el historial (`a_t … a_t3`) y se predice la ventana t+1.
4. Se publica en Grafana: actividad real, predicha y pronosticada por zona, el ranking de todas las zonas y las zonas en alerta (pronóstico ≥ 1,5 × su media).

In [28]:
VENTANA_MS = VENTANA_MIN * 60 * 1000
ETIQUETA = {z["zona"]: f'{z["zona"]} ({z.get("estacion_ejemplo") or ""})'[:48] for z in TOP}

def iniciar_estado():
    global historial, pendientes, eval_suma, eval_n, ult
    historial = defaultdict(lambda: deque(maxlen=4))
    pendientes, ult = {}, {}
    eval_suma, eval_n = 0.0, 0

iniciar_estado()

def features_siguiente(zona, w_ms):
    obj = w_ms + VENTANA_MS
    d = datetime.fromtimestamp(obj / 1000, NY)
    hora, finde = d.hour + d.minute / 60.0, 1 if d.weekday() >= 5 else 0
    media = MEDIA_SLOT.get((zona, finde, d.hour), 0.0)
    h = list(historial[zona])
    a_t3, a_t2, a_t1, a_t = ([media] * (4 - len(h)) + h)[-4:]
    return {"zona": zona, "slot_obj_ms": obj, "a_t": float(a_t), "a_t1": float(a_t1), "a_t2": float(a_t2), "a_t3": float(a_t3),
            "neto_t": float(ult.get(("neto", zona), 0)), "hora_sin": math.sin(2 * math.pi * hora / 24),
            "hora_cos": math.cos(2 * math.pi * hora / 24), "finde_obj": float(finde), "media_slot_obj": float(media)}

def procesar_ventanas(df, batch_id):
    global eval_suma, eval_n
    filas = df.select(F.unix_millis("window.start").alias("w_ms"), "zona", "salidas", "llegadas", "actividad").collect()
    if not filas:
        return
    (df.select(F.lit(batch_id).alias("batchId"), F.col("window.start").alias("ventanaInicio"), "zona", "salidas", "llegadas", "actividad")
       .write.mode("append").parquet(f"{STREAM_OUT}/actividad_ventanas"))
    por_ventana = defaultdict(list)
    for r in filas:
        por_ventana[r["w_ms"]].append(r)
    evaluaciones, predicciones = [], []
    for w_ms in sorted(por_ventana):
        obs = {r["zona"]: r for r in por_ventana[w_ms]}
        for z in IDS_TOP:
            a = obs[z]["actividad"] if z in obs else 0
            ult[("neto", z)] = (obs[z]["llegadas"] - obs[z]["salidas"]) if z in obs else 0
            if (z, w_ms) in pendientes:
                pred = pendientes.pop((z, w_ms))
                evaluaciones.append((z, w_ms, a, pred, abs(pred - a)))
                eval_suma += abs(pred - a); eval_n += 1
                ult[("pred_para", z)] = pred
            historial[z].append(a)
            ult[("a", z)] = a
        feats = [features_siguiente(z, w_ms) for z in IDS_TOP]
        for p in modelo.transform(spark.createDataFrame(feats)).select("zona", "slot_obj_ms", "prediction").collect():
            v = max(0.0, float(p["prediction"]))
            pendientes[(p["zona"], p["slot_obj_ms"])] = v
            ult[("pron", p["zona"])] = v
            ult[("media_sig", p["zona"])] = feats[IDS_TOP.index(p["zona"])]["media_slot_obj"]
            predicciones.append((batch_id, p["zona"], w_ms, p["slot_obj_ms"], v))
        ult["ranking"] = [(i + 1, r["zona"], r["actividad"]) for i, r in enumerate(sorted(por_ventana[w_ms], key=lambda r: -r["actividad"])[:10])]
        ult["total"] = sum(r["actividad"] for r in por_ventana[w_ms])
        ult["ventana_ms"] = w_ms
    if predicciones:
        spark.createDataFrame(predicciones, "batchId long, zona string, ventanaBaseMs long, ventanaObjetivoMs long, prediccion double") \
             .write.mode("append").parquet(f"{STREAM_OUT}/predicciones")
    if evaluaciones:
        spark.createDataFrame(evaluaciones, "zona string, ventanaMs long, real long, prediccion double, errorAbs double") \
             .write.mode("append").parquet(f"{STREAM_OUT}/evaluacion_online")
    L = []
    for z in IDS_TOP:
        lab = {"zona": z, "nombre": ETIQUETA[z]}
        L += [metrica("actividad_observada", ult.get(("a", z)), **lab),
              metrica("actividad_predicha_para_ventana", ult.get(("pred_para", z)), **lab),
              metrica("actividad_pronostico_siguiente", ult.get(("pron", z)), **lab),
              metrica("actividad_media_historica_siguiente", ult.get(("media_sig", z)), **lab)]
    alerta = [z for z in IDS_TOP if ult.get(("pron", z), 0) >= 1.5 * max(ult.get(("media_sig", z), 0), 1.0)]
    L += [metrica("ranking_actividad", a, puesto=f"{p:02d}", zona=z, nombre=ETIQUETA.get(z, z)) for p, z, a in ult["ranking"]]
    L += [metrica("actividad_total_ventana", ult["total"]),
          metrica("pronostico_total_siguiente_top", sum(ult.get(("pron", z), 0) for z in IDS_TOP)),
          metrica("zonas_en_alerta_siguiente", len(alerta)),
          metrica("mae_online", eval_suma / eval_n if eval_n else None),
          metrica("predicciones_evaluadas", eval_n),
          metrica("ventana_actual_info", 1, ventana=datetime.fromtimestamp(ult["ventana_ms"] / 1000, NY).strftime("%Y-%m-%d %H:%M") + " NYC")]
    push("actividad", L)
    print(f"[actividad] batch {batch_id}: {len(por_ventana)} ventana(s) | MAE online = "
          f"{eval_suma / eval_n if eval_n else float('nan'):.3f} ({eval_n} evaluadas)")

actividad = (viajes.filter("isValid").filter(F.col("zona").isNotNull())
    .withWatermark("eventTime", WATERMARK_ACT)
    .groupBy(F.window("eventTime", f"{VENTANA_MIN} minutes"), "zona")
    .agg(F.sum(F.when(F.col("tipoEvento") == "viaje.salida", 1).otherwise(0)).alias("salidas"),
         F.sum(F.when(F.col("tipoEvento") == "viaje.llegada", 1).otherwise(0)).alias("llegadas"))
    .withColumn("actividad", F.col("salidas") + F.col("llegadas")))

#### 5. Consulta `u2z_estado_iot`: docks en vivo, agregados por zona

El feed repite el mismo `last_reported` en varias lecturas. `dropDuplicatesWithinWatermark(["stationId", "timestamp"])` cuenta cada reporte del sensor una sola vez. Se guarda el último estado de **cada estación** y se agrega por zona: bicis, docks, ocupación y estaciones vacías o llenas.

In [29]:
estado_actual = {}

def procesar_estado(df, batch_id):
    filas = df.select("stationId", "zona", "bicisDisponibles", "ebikesDisponibles", "docksDisponibles", "enServicio", "timestamp").collect()
    if not filas:
        return
    df.withColumn("batchId", F.lit(batch_id)).write.mode("append").parquet(f"{STREAM_OUT}/estado_iot")
    for r in filas:
        previo = estado_actual.get(r["stationId"])
        if previo is None or r["timestamp"] >= previo["timestamp"]:
            estado_actual[r["stationId"]] = r.asDict()
    por_zona = defaultdict(lambda: {"bicis": 0, "ebikes": 0, "docks": 0, "est": 0, "vacias": 0, "llenas": 0})
    for e in estado_actual.values():
        if not e["enServicio"] or e["zona"] is None:
            continue
        z = por_zona[e["zona"]]
        z["bicis"] += e["bicisDisponibles"]; z["ebikes"] += e["ebikesDisponibles"] or 0; z["docks"] += e["docksDisponibles"]
        z["est"] += 1; z["vacias"] += int(e["bicisDisponibles"] == 0); z["llenas"] += int(e["docksDisponibles"] == 0)
    L = []
    for zona, z in por_zona.items():
        lab = {"zona": zona, "nombre": ETIQUETA.get(zona, zona)}
        total = z["bicis"] + z["docks"]
        L += [metrica("iot_bicis_zona", z["bicis"], **lab), metrica("iot_docks_zona", z["docks"], **lab),
              metrica("iot_ocupacion_zona_pct", 100.0 * z["bicis"] / total if total else None, **lab),
              metrica("iot_estaciones_vacias_zona", z["vacias"], **lab), metrica("iot_estaciones_llenas_zona", z["llenas"], **lab)]
    L += [metrica("iot_estaciones_reportando", len(estado_actual)), metrica("iot_zonas_reportando", len(por_zona)),
          metrica("iot_estaciones_vacias", sum(z["vacias"] for z in por_zona.values())),
          metrica("iot_estaciones_llenas", sum(z["llenas"] for z in por_zona.values())),
          metrica("iot_reportes_nuevos_batch", len(filas))]
    push("iot", L)
    print(f"[iot] batch {batch_id}: {len(filas)} reportes nuevos | {len(por_zona)} zonas con datos")

estado_iot = (estados.filter("isValid")
    .withWatermark("publishedTime", WATERMARK_IOT)
    .dropDuplicatesWithinWatermark(["stationId", "timestamp"]))

#### 6. Consulta `u2z_calidad`: contrato, latencia y volumen por topic

In [30]:
esquema_min = (StructType().add("tipoEvento", StringType()).add("stationId", StringType()).add("timestamp", LongType())
               .add("publishedAt", LongType()).add("bicisDisponibles", IntegerType()).add("docksDisponibles", IntegerType()))
calidad = (leer_topic(f"{TOPIC_RIDES},{TOPIC_ESTACIONES}")
    .select("topic", F.length("value").alias("bytes"), F.col("value").cast("string").alias("json"))
    .withColumn("e", F.from_json("json", esquema_min)).select("topic", "bytes", "e.*")
    .withColumn("isValid", F.coalesce(F.when(F.col("topic") == TOPIC_RIDES, VALIDO_VIAJE).otherwise(VALIDO_ESTADO), F.lit(False)))
    .withColumn("processedAt", F.unix_millis(F.current_timestamp()))
    .withColumn("latencyMs", F.col("processedAt") - F.col("publishedAt")))

totales_calidad = defaultdict(lambda: {"eventos": 0, "validos": 0, "bytes": 0})

def procesar_calidad(df, batch_id):
    res = (df.groupBy("topic").agg(F.count("*").alias("eventos"), F.sum(F.col("isValid").cast("int")).alias("validos"),
           F.sum("bytes").alias("bytes"), F.avg("latencyMs").alias("lat_avg"),
           F.expr("percentile_approx(latencyMs, 0.95)").alias("lat_p95")).collect())
    if not res:
        return
    df.groupBy("topic").agg(F.count("*").alias("eventos"), F.sum(F.col("isValid").cast("int")).alias("validos"),
                            F.sum("bytes").alias("bytes")).withColumn("batchId", F.lit(batch_id)) \
      .withColumn("ts", F.current_timestamp()).write.mode("append").parquet(f"{STREAM_OUT}/calidad")
    L = []
    for r in res:
        t = totales_calidad[r["topic"]]
        t["eventos"] += r["eventos"]; t["validos"] += r["validos"] or 0; t["bytes"] += r["bytes"] or 0
        lab = {"topic": r["topic"]}
        L += [metrica("calidad_eventos_total", t["eventos"], **lab), metrica("calidad_invalidos_total", t["eventos"] - t["validos"], **lab),
              metrica("calidad_ratio_validos", t["validos"] / t["eventos"], **lab), metrica("calidad_latencia_prom_ms", r["lat_avg"], **lab),
              metrica("calidad_latencia_p95_ms", r["lat_p95"], **lab), metrica("calidad_bytes_por_evento", t["bytes"] / t["eventos"], **lab)]
    push("calidad", L)

#### 7. Arrancar las consultas y el monitor

`REINICIAR = True` borra checkpoints y salidas para empezar de cero. Un checkpoint reutilizado ignora `startingOffsets`.

**Después de esta celda espera 2–3 minutos antes de seguir:** la primera ventana se cierra cuando el watermark pasa su final.

In [31]:
REINICIAR = True

for q in spark.streams.active:
    q.stop()
if REINICIAR:
    shutil.rmtree(CHECKPOINTS, ignore_errors=True); shutil.rmtree(STREAM_OUT, ignore_errors=True)
    iniciar_estado(); estado_actual.clear(); totales_calidad.clear()

q_act = (actividad.writeStream.queryName("u2z_actividad").outputMode("append").foreachBatch(procesar_ventanas)
         .trigger(processingTime="5 seconds").option("checkpointLocation", f"{CHECKPOINTS}/actividad").start())
q_iot = (estado_iot.writeStream.queryName("u2z_estado_iot").outputMode("append").foreachBatch(procesar_estado)
         .trigger(processingTime="10 seconds").option("checkpointLocation", f"{CHECKPOINTS}/estado_iot").start())
q_cal = (calidad.writeStream.queryName("u2z_calidad").foreachBatch(procesar_calidad)
         .trigger(processingTime="5 seconds").option("checkpointLocation", f"{CHECKPOINTS}/calidad").start())

def _num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def _wm(p):
    wm = (p.get("eventTime") or {}).get("watermark")
    return None if not wm or wm.startswith("1970") else datetime.fromisoformat(wm.replace("Z", "+00:00")).timestamp() * 1000

def monitor(cada_s=10):
    while not parar_monitor.is_set():
        L = []
        for q in spark.streams.active:
            p = q.lastProgress
            if not p:
                continue
            lab, ops = {"query": q.name}, (p.get("stateOperators") or [])
            met = ((p.get("sources") or [{}])[0].get("metrics") or {})
            L += [metrica("spark_input_rows_per_s", _num(p.get("inputRowsPerSecond")), **lab),
                  metrica("spark_processed_rows_per_s", _num(p.get("processedRowsPerSecond")), **lab),
                  metrica("spark_batch_duration_ms", (p.get("durationMs") or {}).get("triggerExecution"), **lab),
                  metrica("spark_state_rows", sum(o.get("numRowsTotal", 0) for o in ops), **lab),
                  metrica("spark_rows_dropped_by_watermark", sum(o.get("numRowsDroppedByWatermark", 0) for o in ops), **lab),
                  metrica("spark_duplicados_descartados", sum((o.get("customMetrics") or {}).get("numDroppedDuplicateRows", 0) for o in ops), **lab),
                  metrica("spark_offsets_behind_latest_max", _num(met.get("maxOffsetsBehindLatest")), **lab),
                  metrica("spark_watermark_ms", _wm(p), **lab)]
        if L:
            push("monitor", L)
        parar_monitor.wait(cada_s)

parar_monitor = threading.Event()
threading.Thread(target=monitor, daemon=True).start()
print("Activas:", [q.name for q in spark.streams.active])

Activas: ['u2z_calidad', 'u2z_actividad', 'u2z_estado_iot']


Abre `http://localhost:43000` → **Dashboards** → **Proyecto Sello** → **CitiBike Intelligent Mobility** (dashboard general).

#### 8. Evidencias en vivo (criterios 1-3)

##### 8.0 Diagnóstico rápido (si Grafana no muestra la fila Actividad)

Se esperan ~3 minutos para que el watermark cierre las primeras ventanas de 15 min (el día se reproduce 60× más rápido).

In [32]:
time.sleep(180)
print('Listo: ya debería haber ventanas cerradas')

[iot] batch 1: 2519 reportes nuevos | 105 zonas con datos


[actividad] batch 6: 1 ventana(s) | MAE online = nan (0 evaluadas)


[iot] batch 3: 1602 reportes nuevos | 105 zonas con datos


[actividad] batch 9: 1 ventana(s) | MAE online = 85.507 (15 evaluadas)


[actividad] batch 11: 1 ventana(s) | MAE online = 66.707 (30 evaluadas)


[actividad] batch 13: 1 ventana(s) | MAE online = 52.701 (45 evaluadas)


... (4 outputs left out)

[actividad] batch 23: 1 ventana(s) | MAE online = 49.086 (105 evaluadas)


[actividad] batch 25: 1 ventana(s) | MAE online = 47.675 (120 evaluadas)


[actividad] batch 27: 1 ventana(s) | MAE online = 46.032 (135 evaluadas)


Listo: ya debería haber ventanas cerradas


In [33]:
for q in spark.streams.active:
    p = q.lastProgress or {}
    print(f"{q.name}: batch {p.get('batchId')} | filas {p.get('numInputRows')} | in {p.get('inputRowsPerSecond')} r/s | "
          f"watermark {(p.get('eventTime') or {}).get('watermark')}")
    for o in p.get("stateOperators") or []:
        print(f"   estado: {o.get('operatorName')} filas={o.get('numRowsTotal')} descartadas_watermark={o.get('numRowsDroppedByWatermark')} "
              f"duplicados={(o.get('customMetrics') or {}).get('numDroppedDuplicateRows')}")
print("Errores:", {q.name: str(q.exception()) for q in [q_act, q_iot, q_cal] if q.exception()})

u2z_calidad: batch 36 | filas 2460 | in 492.0 r/s | watermark None
u2z_actividad: batch 28 | filas 2299 | in 262.2 r/s | watermark 2026-07-30T13:54:37.095Z
   estado: stateStoreSave filas=163 descartadas_watermark=0 duplicados=None
u2z_estado_iot: batch 10 | filas 0 | in 0.0 r/s | watermark 2026-11-20T05:48:20.679Z
   estado: dedupeWithinWatermark filas=5708 descartadas_watermark=0 duplicados=0
Errores: {}


##### 8.1 Dato tardío, duplicado e inválido (inyectados a propósito)

In [34]:
def enviar_a_kafka(topic, eventos):
    (spark.createDataFrame([(e["stationId"], json.dumps(e)) for e in eventos], "key string, value string")
     .write.format("kafka").option("kafka.bootstrap.servers", KAFKA).option("topic", topic).save())

wm_ms = _wm(q_act.lastProgress or {}) or (time.time() * 1000)
ahora = int(time.time() * 1000)
est0 = next(e for e, z in EST_ZONA.items() if z == IDS_TOP[0])
tardio = {"tipoEvento": "viaje.salida", "stationId": est0, "stationName": "prueba", "rideId": "PRUEBA-TARDIO", "rideableType": "classic_bike",
          "memberCasual": "member", "origen": "prueba-tardio", "timestamp": int(wm_ms - 2 * 3600 * 1000), "publishedAt": ahora}
dup = {"tipoEvento": "estacion.estado", "stationId": est0, "stationName": "prueba", "capacidad": 40, "lat": None, "lon": None,
       "bicisDisponibles": 7, "ebikesDisponibles": 2, "docksDisponibles": 33, "enServicio": True, "origen": "prueba-duplicado",
       "timestamp": ahora - 60_000, "publishedAt": ahora}
invalido = {**dup, "origen": "prueba-invalido", "bicisDisponibles": -3, "timestamp": ahora - 30_000}
enviar_a_kafka(TOPIC_RIDES, [tardio])
enviar_a_kafka(TOPIC_ESTACIONES, [dup, dup, dup, invalido])
print(f"Inyectados. Watermark de u2z_actividad: {datetime.fromtimestamp(wm_ms / 1000, NY):%Y-%m-%d %H:%M} (NYC); "
      f"viaje tardío: {datetime.fromtimestamp(tardio['timestamp'] / 1000, NY):%Y-%m-%d %H:%M}")

[iot] batch 11: 1766 reportes nuevos | 105 zonas con datos
Inyectados. Watermark de u2z_actividad: 2026-07-30 09:54 (NYC); viaje tardío: 2026-07-30 07:54


In [35]:
time.sleep(25)
caidas = sum(o.get("numRowsDroppedByWatermark", 0) for p in q_act.recentProgress[-8:] for o in p.get("stateOperators", []))
dups = sum((o.get("customMetrics") or {}).get("numDroppedDuplicateRows", 0) for p in q_iot.recentProgress[-8:] for o in p.get("stateOperators", []))
print(f"u2z_actividad  -> filas descartadas por watermark: {caidas}")
print(f"u2z_estado_iot -> duplicados descartados: {dups}")
print(f"u2z_calidad    -> inválidos por topic: { {t: v['eventos'] - v['validos'] for t, v in totales_calidad.items()} }")

[actividad] batch 29: 1 ventana(s) | MAE online = 43.154 (150 evaluadas)


[iot] batch 12: 1 reportes nuevos | 105 zonas con datos


[actividad] batch 33: 1 ventana(s) | MAE online = 40.815 (165 evaluadas)


u2z_actividad  -> filas descartadas por watermark: 1
u2z_estado_iot -> duplicados descartados: 5893
u2z_calidad    -> inválidos por topic: {'citibike-rides': 0, 'citibike-estaciones': 1}


##### 8.2 Checkpoint

In [36]:
for nombre in ("actividad", "estado_iot", "calidad"):
    base = f"{CHECKPOINTS}/{nombre}"
    offs = sorted(int(x) for x in os.listdir(f"{base}/offsets") if x.isdigit())
    commits = [x for x in os.listdir(f"{base}/commits") if x.isdigit()] if os.path.isdir(f"{base}/commits") else []
    print(f"== {nombre}: {len(offs)} micro-lotes en offsets/, {len(commits)} en commits/, state: {os.path.isdir(f'{base}/state')}")
    if offs:
        print("   último offsets/", offs[-1], "->", open(f"{base}/offsets/{offs[-1]}").read().strip().splitlines()[-1])

== actividad: 36 micro-lotes en offsets/, 35 en commits/, state: True
   último offsets/ 35 -> {"citibike-rides":{"0":218359,"1":216715,"2":244658}}
== estado_iot: 14 micro-lotes en offsets/, 14 en commits/, state: True
   último offsets/ 13 -> {"citibike-estaciones":{"0":51522,"1":50530,"2":54134}}
== calidad: 42 micro-lotes en offsets/, 42 en commits/, state: False
   último offsets/ 41 -> {"citibike-rides":{"0":218303,"1":216670,"2":244598},"citibike-estaciones":{"0":52353,"1":51345,"2":55007}}


#### 9. BI y ML en vivo (criterios 6-7)

In [37]:
if not os.path.isdir(f"{STREAM_OUT}/actividad_ventanas"):
    print("Todavía no se cerró ninguna ventana: espera 1-2 minutos y vuelve a correr esta celda (ver 8.0).")
else:
    v = spark.read.parquet(f"{STREAM_OUT}/actividad_ventanas")
    ultima = v.agg(F.max("ventanaInicio")).first()[0]
    print("Ranking de zonas en la última ventana cerrada (hora NYC):")
    (v.filter(F.col("ventanaInicio") == ultima)
      .select(F.date_format("ventanaInicio", "yyyy-MM-dd HH:mm").alias("ventana"), "zona", "salidas", "llegadas", "actividad")
      .orderBy(F.desc("actividad")).show(10, truncate=False))
    print("Pronóstico de actividad para la siguiente ventana:")
    for z in sorted(IDS_TOP, key=lambda z: -ult.get(("pron", z), 0))[:10]:
        p, m = ult.get(("pron", z), 0.0), ult.get(("media_sig", z), 0.0)
        print(f"  {ETIQUETA[z]:48s} pronóstico={p:6.2f}  media={m:6.2f}{'  <- ALERTA' if p >= 1.5 * max(m, 1.0) else ''}")

Ranking de zonas en la última ventana cerrada (hora NYC):


+----------------+------+-------+--------+---------+
|ventana         |zona  |salidas|llegadas|actividad|
+----------------+------+-------+--------+---------+
|2026-07-30 10:00|Z12-15|182    |288     |470      |
|2026-07-30 10:00|Z11-15|221    |244     |465      |
|2026-07-30 10:00|Z12-16|89     |111     |200      |
|2026-07-30 10:00|Z13-15|103    |91      |194      |
|2026-07-30 10:00|Z10-15|90     |97      |187      |
|2026-07-30 10:00|Z12-14|67     |97      |164      |
|2026-07-30 10:00|Z09-15|76     |87      |163      |
|2026-07-30 10:00|Z11-14|68     |94      |162      |
|2026-07-30 10:00|Z10-14|57     |103     |160      |
|2026-07-30 10:00|Z13-16|45     |72      |117      |
+----------------+------+-------+--------+---------+
only showing top 10 rows
Pronóstico de actividad para la siguiente ventana:
  Z12-15 (Lexington Ave & E 26 St)                 pronóstico=383.16  media=335.97
  Z11-15 (E 2 St & Ave B)                          pronóstico=325.13  media=362.72
  Z13-15 (W 44 S

In [38]:
if os.path.isdir(f"{STREAM_OUT}/evaluacion_online"):
    ev_ = spark.read.parquet(f"{STREAM_OUT}/evaluacion_online")
    k = ev_.agg(F.count("*").alias("n"), F.avg("errorAbs").alias("mae"), F.sqrt(F.avg(F.col("errorAbs") ** 2)).alias("rmse")).first()
    mae_test = next(r["MAE"] for r in META["resultados"] if r["modelo"] == META["ganador"])
    print(f"Predicciones evaluadas: {k['n']} | MAE producción = {k['mae']:.3f} | RMSE = {k['rmse']:.3f} | MAE test = {mae_test:.3f}")
else:
    print("Aún no hay predicciones evaluadas: hacen falta al menos 2 ventanas cerradas.")

[iot] batch 14: 1423 reportes nuevos | 105 zonas con datos


Predicciones evaluadas: 165 | MAE producción = 40.815 | RMSE = 58.655 | MAE test = 18.209


#### 10. Estimación de costos y criterios de escalado (criterio 5)

Volumen medido por `u2z_calidad`. `citibike-rides` corre `SPEEDUP` veces más rápido que la realidad, así que su tasa real es la medida dividida entre `SPEEDUP`. `citibike-estaciones` ya sigue **todas** las estaciones, así que su tasa medida es la real. Precios de referencia de S9 (AWS us-east-1): verifícalos antes de sustentar.

In [39]:
import pandas as pd
cal = spark.read.parquet(f"{STREAM_OUT}/calidad")
filas = []
for r in cal.groupBy("topic").agg(F.sum("eventos").alias("eventos"), F.sum("bytes").alias("bytes"),
                                  (F.max(F.unix_timestamp("ts")) - F.min(F.unix_timestamp("ts"))).alias("seg")).collect():
    if not r["seg"]:
        continue
    tasa_demo = r["eventos"] / r["seg"]
    tasa_real = tasa_demo / SPEEDUP if r["topic"] == TOPIC_RIDES else tasa_demo
    b = r["bytes"] / r["eventos"]
    gb_dia = tasa_real * 86400 * b / 1024**3
    filas.append({"topic": r["topic"], "bytes/evento": round(b, 1), "ev/s demo": round(tasa_demo, 2), "ev/s real": round(tasa_real, 2),
                  "GB/día": round(gb_dia, 3), "GB Kafka (7 d)": round(gb_dia * 7, 3), "GB/mes": round(gb_dia * 30, 2)})
tabla_costos = pd.DataFrame(filas)
print(tabla_costos.to_string(index=False))
gb_mes = tabla_costos["GB/mes"].sum() if len(tabla_costos) else 0
computo = 3 * 0.0416 * 730
print(f"\nCómputo: 3 × t3.medium × 730 h = US$ {computo:,.2f}/mes | Almacenamiento: {gb_mes:,.2f} GB × 0,023 = US$ {gb_mes * 0.023:,.2f}/mes "
      f"| Total ≈ US$ {computo + gb_mes * 0.023:,.2f}/mes")

[actividad] batch 35: 1 ventana(s) | MAE online = 39.084 (180 evaluadas)
              topic  bytes/evento  ev/s demo  ev/s real  GB/día  GB Kafka (7 d)  GB/mes
     citibike-rides         303.1     357.54       5.96   0.145           1.017    4.36
citibike-estaciones         422.4      92.34      92.34   3.139          21.970   94.16

Cómputo: 3 × t3.medium × 730 h = US$ 91.10/mes | Almacenamiento: 98.52 GB × 0,023 = US$ 2.27/mes | Total ≈ US$ 93.37/mes


### Cerrar el streaming del notebook

Se detienen las consultas antes de U3, para no tener dos consumidores compitiendo por CPU.

In [40]:
parar_monitor.set()
for q in spark.streams.active:
    q.stop()
print("Consultas detenidas")

Consultas detenidas


26/11/20 06:04:35 ERROR FileFormatWriter: Aborting job 4ebd3e1f-f06e-4914-830f-a0a0567c03ec.
org.apache.spark.SparkException: [SPARK_JOB_CANCELLED] Job 304 cancelled Query u2z_calidad [id = 5b6d0003-1c95-4233-98a2-3e41628428ef, runId = 65c94abd-2005-49b4-af53-86a6823f02a7] was stopped SQLSTATE: XXKDA
	at org.apache.spark.errors.SparkCoreErrors$.sparkJobCancelled(SparkCoreErrors.scala:222)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:3233)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleJobGroupCancelled$7(DAGScheduler.scala:1309)
	at scala.runtime.java8.JFunction1$mcVI$sp.apply(JFunction1$mcVI$sp.scala:18)
	at scala.collection.mutable.HashSet$Node.foreach(HashSet.scala:450)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:376)
	at org.apache.spark.scheduler.DAGScheduler.handleJobGroupCancelled(DAGScheduler.scala:1309)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3542)
	at org.apac

---
# 03 · UNIDAD III — INTEGRACIÓN Y OBSERVABILIDAD

**¿El sistema completo se levanta, se comprueba y se explica?**

`scripts/U3_ejecucion_integrada.py` ejecuta en una sola corrida, en su propio proceso: **compuertas previas** (datos, Kafka, Pushgateway,
Prometheus, artefactos de U1/U2) → **flujo batch** (calidad de Gold) → **flujo streaming** (240 s con el modelo de U2, inyecta un evento
inválido y un viaje tardío) → **publica los KPI** en Pushgateway → Prometheus → Grafana → **manifiesto** por corrida.

### 3.1 Demo: una compuerta detiene la corrida (`--forzar-fallo`)

La compuerta previa falla a propósito: la corrida queda **DETENIDA** antes de arrancar Spark y el manifiesto lo registra. Se ejecuta primero para que la corrida completa (3.2) quede como la última en Grafana.

In [41]:
import subprocess, sys as _sys
r = subprocess.run([_sys.executable, '../scripts/U3_ejecucion_integrada.py', '--forzar-fallo'], capture_output=True, text=True)
print('\n'.join(l for l in r.stdout.splitlines() if not l.startswith((':: ', '\t', 'Ivy', 'confs', 'found'))))
print('Código de salida:', r.returncode, '(1 = detenida, como se esperaba)')

=== U3 · Ejecución integrada · run_id 20261120-060435 ===

1) Compuertas previas
  [OK ] previa    datos CSV: 5 CSV en datos/ (se esperan 5)
  [OK ] previa    kafka: conexión a kafka:9092
  [OK ] previa    pushgateway: http://pushgateway:9091/-/ready
  [OK ] previa    prometheus: http://prometheus:9090/-/ready
  [OK ] previa    artefactos U1: ok
  [OK ] previa    artefactos U2: ok
  [FALLA] previa    prueba de fallo (--forzar-fallo): se pide un archivo que no existe para demostrar que la compuerta detiene la corrida

4) BI/ML: publicar KPI en Pushgateway y confirmar en Prometheus
  [OK ] bi        KPI publicados en Pushgateway: 5 métricas
  [OK ] bi        Prometheus tiene los KPI de ESTA corrida: run_id=20261120-060435
  conteo final de compuertas re-publicado: 8/9

5) Manifiesto: salidas/u3/manifiestos/manifiesto_20261120-060435.json
=== Estado DETENIDA · compuertas 8/9 OK · 0.0 s ===
Código de salida: 1 (1 = detenida, como se esperaba)


### 3.2 Ejecución integrada completa

In [42]:
r = subprocess.run([_sys.executable, '../scripts/U3_ejecucion_integrada.py'], capture_output=True, text=True)
print('\n'.join(l for l in r.stdout.splitlines() if not l.startswith((':: ', '\t', 'Ivy', 'confs', 'found'))))
print('Código de salida:', r.returncode)

=== U3 · Ejecución integrada · run_id 20261120-060451 ===

1) Compuertas previas
  [OK ] previa    datos CSV: 5 CSV en datos/ (se esperan 5)
  [OK ] previa    kafka: conexión a kafka:9092
  [OK ] previa    pushgateway: http://pushgateway:9091/-/ready
  [OK ] previa    prometheus: http://prometheus:9090/-/ready
  [OK ] previa    artefactos U1: ok
  [OK ] previa    artefactos U2: ok

2) Flujo batch (U1): compuertas de calidad sobre Gold
  [OK ] batch     existencia: 73,953 filas zona-hora en 99 zonas
  [OK ] batch     completitud: 0 filas sin zona u hora
  [OK ] batch     rango: 0 filas con salidas o llegadas negativas
  [OK ] batch     plausibilidad: Silver conserva >= 95 % de Bronze: 99.57 %
  [OK ] batch     plausibilidad: Gold cuadra con Silver: actividad Gold 9,943,694 vs 2 x Silver 9,943,694

3) Flujo streaming (U2): 240 s leyendo citibike-rides con el modelo de U2
  inyectados: 1 evento inválido y 1 viaje tardío

  Compuertas de calidad del streaming
  [OK ] streaming sin errores 

### No regresión: comparar las dos últimas corridas OK

In [43]:
import glob, json as _json
oks = [p for p in sorted(glob.glob('../salidas/u3/manifiestos/manifiesto_2*.json'))
       if _json.load(open(p, encoding='utf-8'))['estado'] == 'OK']
if len(oks) >= 2:
    r = subprocess.run([_sys.executable, '../scripts/U3_comparar_manifiestos.py', oks[-2], oks[-1]], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print('Hace falta una segunda corrida OK para comparar. Vuelve a ejecutar la celda de U3 y luego esta.')
    print('Evidencia guardada de la entrega:')
    print(open('../evidencias/no_regresion.md', encoding='utf-8').read())

Hace falta una segunda corrida OK para comparar. Vuelve a ejecutar la celda de U3 y luego esta.
Evidencia guardada de la entrega:
# No regresión: 20261120-002509 (antes) vs 20261120-002852 (después)

| Comprobación | Antes | Después | ¿Regresión? |
|---|---:|---:|:---:|
| Estado de la corrida | OK | OK | no |
| Compuertas OK | 21/21 | 21/21 | no |
| Eventos leídos de citibike-rides | 67,194 | 50,852 | — |
| Ratio de eventos válidos | 0.99999 | 0.99998 | no |
| Ventanas de 15 min cerradas | 12 | 12 | no |
| Descartados por watermark (prueba) | 1 | 1 | — |
| Latencia p50 (ms) | 2,527 | 2,493 | no |
| Latencia p95 (ms) | 4,776 | 4,774 | no |
| Micro-lote p50 (ms) | 340 | 352 | no |
| Micro-lote p95 (ms) | 660.05 | 754 | no |
| MAE en vivo (modelo U2) | 33.3972 | 16.3473 | no |
| MAE en test (referencia) | 18.2092 | 18.2092 | — |
| Duración total (s) | 208.1 | 208.3 | — |



---
# 04 · Cierre: dashboard general y streaming en vivo

1. Regenera el **dashboard general** de Grafana con los resultados de U1 de esta corrida (Grafana lo recarga solo en ≤ 30 s).
2. Deja el streaming de U2 corriendo **en segundo plano** (sigue vivo aunque cierres el notebook) para que el dashboard muestre datos en vivo.
   Para detenerlo: `docker compose exec spark python scripts/streaming_en_segundo_plano.py --detener`.

In [44]:
r = subprocess.run([_sys.executable, '../scripts/generar_dashboard_general.py'], capture_output=True, text=True)
print(r.stdout or r.stderr)
subprocess.run([_sys.executable, '../scripts/streaming_en_segundo_plano.py', '--detener'])
subprocess.Popen([_sys.executable, '../scripts/streaming_en_segundo_plano.py'], start_new_session=True,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('Streaming en segundo plano iniciado (log: /tmp/u2_02_stream.log).')
print('Dashboard general: http://localhost:43000/d/citibike-general-cristhian  (admin / admin)')

Dashboard escrito en infra/grafana/provisioning/dashboards/json/dashboard_general_cristhian.json · 33 paneles · periodo U1 julio 2026 (viajes del 2026-07-01 al 2026-07-31; Gold incluye llegadas hasta 2026-08-01 03:00)

No había streaming en segundo plano.
Streaming en segundo plano iniciado (log: /tmp/u2_02_stream.log).
Dashboard general: http://localhost:43000/d/citibike-general-cristhian  (admin / admin)


## Conclusiones

1. **Pasado (U1):** pocas zonas concentran la mayor parte de la actividad (las 10 primeras suman más de la mitad) y el modelo de MLlib
   detecta las horas de alta actividad con F1 ≈ 0,94.
2. **Futuro (U2):** el modelo t+1 por zona mejora a la persistencia y corre en vivo sobre Kafka + Spark Structured Streaming, junto con la
   disponibilidad real de bicicletas del feed GBFS (IoT).
3. **Funcionamiento (U3):** todo se levanta con un comando, pasa sus compuertas, publica sus KPI en Grafana, deja un manifiesto por corrida
   y se comprueba que no hay regresión.